In [129]:
import cobra
import pandas as pd
from cobra.io import load_model
from cobra.io import load_json_model, save_json_model, load_matlab_model, save_matlab_model, read_sbml_model, write_sbml_model
from cobra import Model, Reaction, Metabolite, Gene
import numpy as np

## Data extraction

In [2]:
# open excel file containing the identifiers list\n
file_path ='C:\\\\Users\\\\adam-1y9pxi6q95ggpmx\\\\Documents\\\\PhD\\\\10-19 Research\\\\11 Data\\\\11.13 Biolog_data\\\\240927_Metabolite_identifier.xlsx'
metabolite_data = pd.read_excel(file_path)

In [3]:
# extract identifier of the plate PM01\n",
metabolite_df = metabolite_data[metabolite_data['Plate'] == 'PM01']
metabolite_df

,Plate,Location,Chemical,CAS ID,KEGG ID,MoA
0,PM01,A01,Negative Control,Negative Control,NaN,"C-Source, negative control"
1,PM01,A02,L-Arabinose,CAS 87-72-9,C00259,"C-Source, carbohydrate"
2,PM01,A03,N-Acetyl-D-Glucosamine,CAS 7512-17-6,C00140,"C-Source, carbohydrate"
3,PM01,A04,D-Saccharic acid,CAS 576-42-1,C00818,"C-Source, carboxylic acid"
4,PM01,A05,Succinic acid,CAS 6106-21-4,C00042,"C-Source, carboxylic acid"
...,...,...,...,...,...,...
91,PM01,H08,Pyruvic acid,CAS 113-24-6,C00022,"C-Source, carboxylic acid"
92,PM01,H09,L-Galactonic acid-g-Lactone,CAS 1668-08-2,C01115,"C-Source, carboxylic acid"
93,PM01,H10,D-Galacturonic acid,CAS 91510-62-2,C00333,"C-Source, carboxylic acid"
94,PM01,H11,b-Phenylethylamine,CAS 156-28-5,C05332,"C-Source, amine"


In [4]:
# open excel file containing the OD600 values list\n
file_path ='C:\\\\Users\\\\adam-1y9pxi6q95ggpmx\\\\Documents\\\\PhD\\\\10-19 Research\\\\11 Data\\\\11.13 Biolog_data\\\\240927_Biolog_OD.xlsx'
biolog_data = pd.read_excel(file_path)
biolog_data

,A1,B1,C1,D1,E1,F1,G1,H1,A2,B2,...,G11,H11,A12,B12,C12,D12,E12,F12,G12,H12
0,0.2568,0.2702,0.2835,0.2581,0.2759,0.2682,0.2786,0.2873,0.4229,0.2689,...,0.761,0.2482,0.2414,0.2416,0.2392,0.2435,0.2534,0.2413,0.9939,0.2695


In [ ]:
# create a list with only OD above the threshhold 
# we used a threshold corresponding ot 1.2 of the control 
filtered_values = [element for element in biolog_data.values.flatten()]
filtered_values

In [ ]:
# Create a dictionary with only OD values above the threshold
filtered_dict = {col: biolog_data[col].tolist() 
                 for col in biolog_data.columns }
print(filtered_dict)

In [ ]:
# Create a new dictionary with updated keys that contain an additional zero for single digits
updated_dict = {}

for key, value in filtered_dict.items():
    # Split the letter and the number part of the key
    letter = key[0]
    number = key[1:]
    
    # Check if the number is a single digit and add a leading zero if necessary
    if len(number) == 1:
        number = '0' + number
    
    # Recreate the key with the letter and the updated number
    new_key = letter + number
    
    # Add the new key and its value to the updated dictionary
    updated_dict[new_key] = value

# Print the updated dictionary
updated_dict

In [ ]:
# Create a DataFrame containing the location, chemical name, OD, and KEGG_ID
df_data = {'Location': [], 'Chemical_name': [], 'OD': [], 'KEGG_ID': []}
biolog_df = pd.DataFrame(df_data)

# Print the DataFrame
biolog_df

In [ ]:
# Create an empty list to store the rows
rows_list = []

# Iterate over the updated_dict and build the rows
for key in updated_dict:
    row = {
        'Location': key,
        'OD': updated_dict[key][0],
        'Chemical_name': metabolite_df.loc[metabolite_df['Location'] == key, 'Chemical'].values[0],
        'KEGG_ID': metabolite_df.loc[metabolite_df['Location'] == key, 'KEGG ID'].values[0]
    }
    rows_list.append(row)

# Convert the list of rows into a DataFrame
biolog_df = pd.DataFrame(rows_list)

# Print the DataFrame
print(biolog_df)

In [ ]:
biolog_df.to_excel('C:\\\\Users\\\\adam-1y9pxi6q95ggpmx\\\\Documents\\\\PhD\\\\10-19 Research\\\\11 Data\\\\11.13 Biolog_data\\\\biolog_df.xlsx', index=False)

### Addition of bigg ids to dataframe

In [34]:
model = read_sbml_model('C:\\\\Users\\\\adam-1y9pxi6q95ggpmx\\\\Documents\\\\PhD\\\\10-19 Research\\\\11 Data\\\\11.09 Models\\\\Manual_curation\\\\Biomass_equation\\\\GAM_NGAM_update.sbml')

In [ ]:
# Create a new column 'BIGG_ID' in biolog_df to store concatenated results
biolog_df['BIGG_ID'] = None  # Initialize with None values

# Iterate over metabolites in the model and update biolog_df with the corresponding bigg_ID
for index, kegg_code in biolog_df['KEGG_ID'].items():
    bigg_ids = []  # Initialize an empty list to store all matching bigg_IDs
    for metabolite in model.metabolites:
        # Check if the metabolite has a 'kegg.compound' annotation
        if 'kegg.compound' in metabolite.annotation:
            kegg_annotation = metabolite.annotation['kegg.compound']
            
            # If the annotation is a list, check if kegg_code is in the list
            if isinstance(kegg_annotation, list) and kegg_code in kegg_annotation:
                bigg_ids.append(metabolite.id)
            # If it's a string, just check for equality
            elif isinstance(kegg_annotation, str) and kegg_annotation == kegg_code:
                bigg_ids.append(metabolite.id)

    # Store the concatenated bigg_IDs as a string in biolog_df
    if bigg_ids:
        biolog_df.at[index, 'BIGG_ID'] = ', '.join(bigg_ids)

# Display the updated biolog_df
print(biolog_df)

### Addition of exchange reaction id to dataframe

In [ ]:
# Iterate over each element in biolog_df.BIGG_ID
for index, element in biolog_df['BIGG_ID'].items():
    
    # Skip if element is None
    if element is None:
        continue
    
    # Check if the element is a string, and if so, split it by commas
    if isinstance(element, str):
        bigg_ids = element.split(', ')
    else:
        bigg_ids = element  # If it's already a list, keep it as is
    
    # List to store the exchange reactions for this row
    exchange_reactions = []
    
    # Iterate over each metabolite ID in the bigg_ids list
    for met in bigg_ids:
        if '_e' in met:  # Only process metabolites with '_e'
            metabolite = model.metabolites.get_by_id(met)
            related_reactions = list(metabolite.reactions)
            # Iterate over the related reactions
            for reaction in related_reactions:
                if 'EX_' in reaction.id:  # If the reaction ID contains 'EX_'
                    exchange_reactions.append(reaction.id)
    
    # If there are any exchange reactions, store them in the DataFrame
    if exchange_reactions:
        biolog_df.at[index, 'model_exchange_reaction'] = ', '.join(exchange_reactions)

# Display the updated biolog_df
print(biolog_df)

## Addition of reaction to the model

In [191]:
model = read_sbml_model('C:\\Users\\adam-1y9pxi6q95ggpmx\\Documents\\PhD\\10-19 Research\\11 Data\\11.09 Models\\Manual_curation\\Biolog\\241009_updated_exchange.sbml')

set the reactions bounds to zero to reset the medium

In [96]:
# List of metabolites to simulate yeast extract and M9 minimal medium with biolog component
yeast_extract_components = [
    'EX_ala__L_e',  # L-alanine
    'EX_arg__L_e',  # L-arginine
    'EX_asp__L_e',  # L-aspartate
    'EX_glu__L_e',  # L-glutamate
    'EX_his__L_e',  # L-histidine
    'EX_ile__L_e',  # L-isoleucine
    'EX_leu__L_e',  # L-leucine
    'EX_lys__L_e',  # L-lysine
    'EX_met__L_e',  # L-methionine
    'EX_phe__L_e',  # L-phenylalanine
    'EX_ser__L_e',  # L-serine
    'EX_thr__L_e',  # L-threonine
    'EX_trp__L_e',  # L-tryptophan
    'EX_val__L_e',  # L-valine
    'EX_btn_e',     # Biotin
    'EX_thm_e',     # Thiamin
    'EX_nac_e',     # Nicotinate (Niacin)
    'EX_pnto__R_e', # Pantothenate
    'EX_ribflv_e',  # Riboflavin
    'EX_ade_e',     # Adenine
    'EX_gua_e',     # Guanine
    'EX_ura_e',     # Uracil
    'EX_cys_e',     # Cytosine
    'EX_mg2_e',     # Magnesium ion
    'EX_k_e',       # Potassium ion
    'EX_h2o_e',      # Phosphate
    'EX_ca2_e',    # Ca2+ (Calcium ion)
    'EX_cl_e',     # Cl- (Chloride ion)
    'EX_cobalt2_e', # Co2+ (Cobalt ion)
    'EX_cu2_e',    # Cu2+ (Copper ion)
    'EX_fe2_e',    # Fe2+ (Ferrous iron)
    'EX_fe3_e',    # Fe3+ (Ferric iron)
    'EX_h_e',      # H+ (Protons)
    'EX_k_e',      # K+ (Potassium ion)
    'EX_mg2_e',    # Mg2+ (Magnesium ion)
    'EX_mn2_e',    # Mn2+ (Manganese ion)
    'EX_mobd_e',   # Molybdate (MoO4 2-)
    'EX_na1_e',    # Na+ (Sodium ion)
    'EX_nh4_e',    # Ammonium (NH4+)
    'EX_ni2_e',    # Ni2+ (Nickel ion)
    'EX_o2_e',     # O2 (Oxygen)
    'EX_pi_e',     # Phosphate (PO4 3-)
    'EX_so4_e',    # Sulfate (SO4 2-)
    'EX_zn2_e'     # Zn2+ (Zinc ion)
]
    

# Set the bounds for each metabolite to allow uptake
for metabolite in yeast_extract_components:
    if metabolite in model.reactions:
        model.reactions.get_by_id(metabolite).lower_bound = -0.5  # Allow unlimited uptake
    else:
        print(f"Metabolite {metabolite} not found in the model")

Metabolite EX_ni2_e not found in the model


In [ ]:
model.summary()

### Simulating growth and importing data for the dataframe

In [192]:
file_path ='C:\\\\Users\\\\adam-1y9pxi6q95ggpmx\\\\Documents\\\\PhD\\\\10-19 Research\\\\11 Data\\\\11.13 Biolog_data\\\\241002_biolog_df.xlsx'
biolog_df = pd.read_excel(file_path)


In [193]:
# itterating over the existant exchanges reaction from the biolog data

# Create a new column in biolog_df to store concatenated results
biolog_df['model_growth'] = None  # Initialize with None values
biolog_df['model_growth_value'] = None  # Initialize with None values

# Iterate over the exchange reactions from the Biolog data
for index, ex_reactions in biolog_df['model_exchange_reaction'].items():
    if pd.notna(ex_reactions): 
        try:
            copy_of_model = model.copy()
            reaction = copy_of_model.reactions.get_by_id(ex_reactions)
            reaction.bounds = -10.0, 0.0
            
            # Optimize the model and store the growth solution
            biolog_df.at[index, 'model_growth_value'] = copy_of_model.slim_optimize() 
            
            # Set 'yes' or 'no' based on the growth value
            if copy_of_model.slim_optimize()  > 0.009:
                biolog_df.at[index, 'model_growth'] = 'yes'
            else:
                biolog_df.at[index, 'model_growth'] = 'no'
            
            
        
        except KeyError:
            # If the reaction ID is not found in the model, set NaN
            biolog_df.at[index, 'model_growth'] = np.nan
            biolog_df.at[index, 'model_growth_value'] = np.nan



In [194]:
# Create a new column in biolog_df to store concatenated results
biolog_df['experimental_growth'] = None  # Initialize with None values
for index, od_values in biolog_df['OD'].items():
    if od_values >= 0.33384 :
        biolog_df.at[index, 'experimental_growth'] = 'yes'
    else:
        biolog_df.at[index, 'experimental_growth'] = 'no'

In [ ]:
# Replace NaN values in the 'model_growth_value' column with 'no'
biolog_df['model_growth'].fillna('no', inplace=True)


### save excel file

In [198]:
biolog_df.to_excel('C:\\\\Users\\\\adam-1y9pxi6q95ggpmx\\\\Documents\\\\PhD\\\\10-19 Research\\\\11 Data\\\\11.13 Biolog_data\\\\241009_growth_biolog_df.xlsx', index=False)

### save model

In [100]:
sbml_filename = "C:\\Users\\adam-1y9pxi6q95ggpmx\\Documents\\PhD\\10-19 Research\\11 Data\\11.09 Models\\Manual_curation\\Biolog\\241009_updated_exchange.sbml"
cobra.io.write_sbml_model(model, sbml_filename)

### confusion matrix

In [199]:
from sklearn.metrics import confusion_matrix

# Sample data: Replace with your actual data
# Observed growth (experimental): 1 for growth, 0 for no growth
observed_growth = biolog_df['experimental_growth'].apply(lambda x: 1 if x == 'yes' else 0)
# Predicted growth (model): 1 for growth, 0 for no growth
predicted_growth = biolog_df['model_growth'].apply(lambda x: 1 if x == 'yes' else 0)

# Calculate the confusion matrix
cm = confusion_matrix(observed_growth, predicted_growth, labels=[1, 0])
cm_df = pd.DataFrame(cm, index=['Observed Growth', 'Observed No Growth'], columns=['Predicted Growth', 'Predicted No Growth'])

print(cm_df)

                    Predicted Growth  Predicted No Growth
Observed Growth                    4                   19
Observed No Growth                21                   52
